In [1]:
import os, numpy as np, pandas as pd

print("Input folder contents:", os.listdir("../input"))




Input folder contents: ['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
usecols = [
    "fare_amount",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]
dtypes = {
    "fare_amount": np.float32,
    "pickup_longitude": np.float32,
    "pickup_latitude": np.float32,
    "dropoff_longitude": np.float32,
    "dropoff_latitude": np.float32,
    "passenger_count": np.int8,
}
train_path = "../input/train.csv"
train_df = pd.read_csv(
    train_path,
    usecols=usecols,
    dtype=dtypes,
    parse_dates=["pickup_datetime"],
    nrows=1_000_000,  # reduced from 5 M rows for speed
)




In [3]:
train_df["abs_diff_longitude"] = (
    train_df["dropoff_longitude"] - train_df["pickup_longitude"]
).abs()
train_df["abs_diff_latitude"] = (
    train_df["dropoff_latitude"] - train_df["pickup_latitude"]
).abs()

train_df["manhattan_distance"] = (
    train_df["abs_diff_longitude"] + train_df["abs_diff_latitude"]
)


def haversine_vec(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in miles."""
    p = np.pi / 180.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))  # Earth radius ≈3958.8 miles


train_df["distance_miles"] = haversine_vec(
    train_df["pickup_latitude"],
    train_df["pickup_longitude"],
    train_df["dropoff_latitude"],
    train_df["dropoff_longitude"],
)

train_df["hour"] = train_df["pickup_datetime"].dt.hour




/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sin
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in cos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arcsin
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
used_cols = [
    "fare_amount",
    "abs_diff_longitude",
    "abs_diff_latitude",
    "manhattan_distance",
    "distance_miles",
    "passenger_count",
    "hour",
]
train_df = train_df.dropna(subset=used_cols)

train_df = train_df[
    (train_df["abs_diff_longitude"] < 3.0)
    & (train_df["abs_diff_latitude"] < 3.0)
    & (train_df["fare_amount"] > 0)
    & (train_df["fare_amount"] <= 200)  # upper fare bound
    & (train_df["passenger_count"] <= 9)
    & (train_df["distance_miles"] > 0)
    & (train_df["distance_miles"] <= 100)  # reasonable distance bound
]

print(f"Rows after cleaning: {len(train_df)}")




Rows after cleaning: 969256


In [5]:
from sklearn.model_selection import train_test_split

y = train_df["fare_amount"].values.astype(np.float32)
X = train_df[
    [
        "abs_diff_longitude",
        "abs_diff_latitude",
        "manhattan_distance",
        "distance_miles",
        "passenger_count",
        "hour",
    ]
].astype(np.float32)

train_X_df, val_X_df, train_y, val_y = train_test_split(
    X, y, test_size=0.2, random_state=42
)




In [6]:
train_X = train_X_df.values  # shape (n_samples, 6), float32 already
val_X = val_X_df.values

print("Design matrix shapes:", train_X.shape, val_X.shape)




Design matrix shapes: (775404, 6) (193852, 6)


In [7]:
from sklearn.experimental import enable_hist_gradient_boosting  # noqa
from sklearn.ensemble import HistGradientBoostingRegressor

gbr = HistGradientBoostingRegressor(
    max_iter=400,  # equivalent to n_estimators
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)
gbr.fit(train_X, train_y)

print("HistGradientBoostingRegressor trained.")




/usr/local/lib/python3.11/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:16: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


HistGradientBoostingRegressor trained.


In [8]:
from sklearn.metrics import mean_squared_error

val_pred = gbr.predict(val_X)
rmse = np.sqrt(mean_squared_error(val_y, val_pred))
print(f"Validation RMSE: {rmse:.4f}")




Validation RMSE: 4.2276


In [9]:
test_path = "../input/test.csv"
test_usecols = [
    "key",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]
test_dtypes = {
    "pickup_longitude": np.float32,
    "pickup_latitude": np.float32,
    "dropoff_longitude": np.float32,
    "dropoff_latitude": np.float32,
    "passenger_count": np.int8,
}
test_df = pd.read_csv(
    test_path,
    usecols=test_usecols,
    dtype=test_dtypes,
    parse_dates=["pickup_datetime"],
)

test_df["abs_diff_longitude"] = (
    test_df["dropoff_longitude"] - test_df["pickup_longitude"]
).abs()
test_df["abs_diff_latitude"] = (
    test_df["dropoff_latitude"] - test_df["pickup_latitude"]
).abs()
test_df["manhattan_distance"] = (
    test_df["abs_diff_longitude"] + test_df["abs_diff_latitude"]
)

test_df["distance_miles"] = haversine_vec(
    test_df["pickup_latitude"],
    test_df["pickup_longitude"],
    test_df["dropoff_latitude"],
    test_df["dropoff_longitude"],
)
test_df["hour"] = test_df["pickup_datetime"].dt.hour

test_features = test_df[
    [
        "abs_diff_longitude",
        "abs_diff_latitude",
        "manhattan_distance",
        "distance_miles",
        "passenger_count",
        "hour",
    ]
].astype(np.float32)

test_predictions = gbr.predict(test_features.values)
# Ensure predictions are non‑negative
test_predictions = np.clip(test_predictions, 0, None)
test_predictions = np.round(test_predictions, 2)

submission = pd.DataFrame({"key": test_df["key"], "fare_amount": test_predictions})
submission.to_csv("submission.csv", index=False)
print("Submission written. Files in cwd:", os.listdir("."))

Submission written. Files in cwd: ['submission.csv', 'new-york-city-taxi-fare-prediction', 'new-york-city-taxi-fare-prediction_rishabh254_nyc-ola_v11_C1.ipynb']
